In [75]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import pickle
import json
import os
from syto.data.pseudobulk_hdf5_utils import PseudobulkHDF5Reader
from typing import Dict
import h5py

### Compare feature matrices

In [64]:
methylbert_soft_labels_data_path = "/mnt/data/syto_experiments/mlflow/198121009408531961/e21d054a7dc04dd2a7e8de4231efaa56/artifacts/pseudobulk_generation/pseudobulk.h5"
methylbert_hard_labels_data_path = "/mnt/data/syto_experiments/mlflow/198121009408531961/a3f1f111ffc14085bc70c8bf60a78e4d/artifacts/pseudobulk_generation/pseudobulk.h5"

In [65]:
reader_soft = PseudobulkHDF5Reader(methylbert_soft_labels_data_path)
reader_hard = PseudobulkHDF5Reader(methylbert_hard_labels_data_path)
splits = ["train", "valid", "test"]

In [66]:

def get_pure_profiles(reader, splits, num_input_labels):
    pure_profiles: Dict[str, np.ndarray] = {}
    for split_name in splits:
        matrix = reader.read_pure_feature_matrix(
            split_name, num_pred_classes=num_input_labels
        )
        pure_profiles[split_name] = matrix

    return pure_profiles

In [67]:
soft_pure = get_pure_profiles(reader_soft, splits, 39)
hard_pure = get_pure_profiles(reader_hard, splits, 40)

In [76]:
with h5py.File(methylbert_hard_labels_data_path, "r") as f:
    _ct = f["parameters/cell_types_mapping"][...]
labels_dict = {int(i): n.decode() for n, i in _ct}
labels_dict_reversed = {v: k for k, v in labels_dict.items()}
n_labels = len(labels_dict)
print(n_labels, "cell types")

39 cell types


In [73]:
def compare_model_predictions_v3(
    data1,
    data2,
    true_props_array=None,
    class_name_key="dmr_ctype",
    class_idx_key="dmr_ctype_label",
    save_path=None,
    model_names=("Model 1", "Model 2"),
    figsize=(16, 6),
    highlight_color="red",
    highlight_lw=1.8,
    fontsize=12,
    diff_cmap="RdBu_r",
):
    """
    Compare prediction matrices from two models as three heatmaps in a row:
        [Model 1]  [Model 2]  [Difference (M2 − M1)]

    Models may have different numbers of prediction columns (e.g. 39 vs 40).
    Each model's heatmap shows ALL of its own columns. The difference heatmap
    is computed only over the intersection of columns present in both models.

    Rows (DMR groups) are union-aligned: every group present in either model
    appears as a row; missing rows are NaN-masked.

    If `true_props_array` is provided, the row and column of the class with the
    largest ground-truth value are highlighted with a colored rectangle in every
    panel where that column exists.
    """
    # ── 1. Convert to DataFrames ──
    df1 = pd.DataFrame(data1)
    df2 = pd.DataFrame(data2)

    # ── 2. Master ROW index: union of row labels from both models ──
    all_labels = pd.concat(
        [
            df1[[class_idx_key, class_name_key]],
            df2[[class_idx_key, class_name_key]],
        ]
    ).drop_duplicates()
    all_labels = all_labels.sort_values(class_idx_key).reset_index(drop=True)

    master_row_indices = all_labels[class_idx_key].tolist()
    master_row_names = all_labels[class_name_key].tolist()
    n_rows = len(master_row_indices)

    # ── 3. Per-model COLUMN indices: each model keeps its own columns ──
    def get_pred_cols(df):
        """Return sorted (col_indices, col_names, col_strings) for one model."""
        raw_cols = [
            c for c in df.columns if c.startswith("prediction_") and c.endswith("_wavg")
        ]
        # Sort by the numeric index, not lexicographically
        raw_cols_with_idx = [(int(c.split("_")[1]), c) for c in raw_cols]
        raw_cols_with_idx.sort(key=lambda x: x[0])
        col_indices = [idx for idx, _ in raw_cols_with_idx]
        pred_cols = [col for _, col in raw_cols_with_idx]
        # Build index→name map from the model's own label data
        idx_to_name = dict(zip(df[class_idx_key], df[class_name_key]))
        # For columns whose index isn't in the model's rows (e.g. background),
        # fall back to a generic name
        col_names = [idx_to_name.get(i, f"class_{i}") for i in col_indices]
        return col_indices, col_names, pred_cols

    col_idx1, col_names1, pred_cols1 = get_pred_cols(df1)
    col_idx2, col_names2, pred_cols2 = get_pred_cols(df2)
    n_cols1 = len(col_idx1)
    n_cols2 = len(col_idx2)

    # ── 4. Align each model's rows to the master row index ──
    def align_rows(df, master_idx, pred_cols):
        aligned = df.set_index(class_idx_key).reindex(master_idx)
        # Ensure all requested pred_cols exist (fills missing with NaN)
        for col in pred_cols:
            if col not in aligned.columns:
                aligned[col] = np.nan
        return aligned[pred_cols].values

    mat1 = align_rows(df1, master_row_indices, pred_cols1)  # (n_rows, n_cols1)
    mat2 = align_rows(df2, master_row_indices, pred_cols2)  # (n_rows, n_cols2)

    # ── 5. Difference over INTERSECTING columns only ──
    shared_col_idx = sorted(set(col_idx1) & set(col_idx2))
    shared_col_names = []
    # Column positions in each model's matrix for the shared subset
    pos_in_1 = []
    pos_in_2 = []
    for idx in shared_col_idx:
        pos_in_1.append(col_idx1.index(idx))
        pos_in_2.append(col_idx2.index(idx))
        # Use name from model 1 (they should agree for shared columns)
        shared_col_names.append(col_names1[col_idx1.index(idx)])

    n_shared = len(shared_col_idx)
    diff_mat = mat2[:, pos_in_2] - mat1[:, pos_in_1]  # (n_rows, n_shared)

    # ── 6. Diagonal scores (over shared columns only, matched by row index) ──
    diag1 = np.full(n_rows, np.nan)
    diag2 = np.full(n_rows, np.nan)
    for i, row_idx in enumerate(master_row_indices):
        if row_idx in col_idx1:
            j = col_idx1.index(row_idx)
            diag1[i] = mat1[i, j]
        if row_idx in col_idx2:
            j = col_idx2.index(row_idx)
            diag2[i] = mat2[i, j]

    # ── 7. Ground-truth highlight indices (per-panel) ──
    gt_row = None
    gt_col1 = None  # column index in mat1
    gt_col2 = None  # column index in mat2
    gt_col_diff = None  # column index in diff_mat
    if true_props_array is not None:
        true_props_array = np.asarray(true_props_array, dtype=float)
        if np.any(~np.isnan(true_props_array)):
            gt_row = int(np.nanargmax(true_props_array))
            gt_label = master_row_indices[gt_row]
            if gt_label in col_idx1:
                gt_col1 = col_idx1.index(gt_label)
            if gt_label in col_idx2:
                gt_col2 = col_idx2.index(gt_label)
            if gt_label in shared_col_idx:
                gt_col_diff = shared_col_idx.index(gt_label)

    def draw_highlight(ax, row_idx, col_idx, n_rows_, n_cols_):
        """Outline a row and/or column on a heatmap."""
        if row_idx is not None:
            ax.add_patch(
                Rectangle(
                    (0, row_idx),
                    n_cols_,
                    1,
                    fill=False,
                    edgecolor=highlight_color,
                    linewidth=highlight_lw,
                    zorder=5,
                    clip_on=False,
                )
            )
        if col_idx is not None:
            ax.add_patch(
                Rectangle(
                    (col_idx, 0),
                    1,
                    n_rows_,
                    fill=False,
                    edgecolor=highlight_color,
                    linewidth=highlight_lw,
                    zorder=5,
                    clip_on=False,
                )
            )

    # ── 8. Figure layout ──
    # Panel widths proportional to column counts so cells stay roughly square
    sns.set_theme(style="white")
    fig = plt.figure(figsize=figsize)

    left, right = 0.06, 0.985
    bottom, top = 0.08, 0.93
    total_w = right - left
    wspace = 0.02
    raw_ratios = np.array([n_cols1, n_cols2, n_shared], dtype=float)
    widths = (total_w - 2 * wspace) * raw_ratios / raw_ratios.sum()
    lefts = [left]
    for w in widths[:-1]:
        lefts.append(lefts[-1] + w + wspace)
    height = top - bottom

    ax1 = fig.add_axes([lefts[0], bottom, widths[0], height])
    ax2 = fig.add_axes([lefts[1], bottom, widths[1], height], sharey=ax1)
    ax3 = fig.add_axes([lefts[2], bottom, widths[2], height], sharey=ax1)

    # Shared color scale for the two prediction heatmaps
    vmax_pred = np.nanmax([np.nanmax(mat1), np.nanmax(mat2)])
    vmin_pred = 0.0
    pred_cmap = "viridis"

    # --- Panel 1: Model 1 ---
    sns.heatmap(
        mat1,
        ax=ax1,
        cmap=pred_cmap,
        vmin=vmin_pred,
        vmax=vmax_pred,
        cbar=False,
        mask=np.isnan(mat1),
        xticklabels=False,
        yticklabels=False,
    )
    ax1.set_title(model_names[0], fontsize=fontsize, fontweight="bold")
    ax1.tick_params(left=False, bottom=False)
    draw_highlight(ax1, gt_row, gt_col1, n_rows, n_cols1)

    # --- Panel 2: Model 2 ---
    sns.heatmap(
        mat2,
        ax=ax2,
        cmap=pred_cmap,
        vmin=vmin_pred,
        vmax=vmax_pred,
        cbar=False,
        mask=np.isnan(mat2),
        xticklabels=False,
        yticklabels=False,
    )
    ax2.set_title(model_names[1], fontsize=fontsize, fontweight="bold")
    ax2.tick_params(left=False, bottom=False)
    draw_highlight(ax2, gt_row, gt_col2, n_rows, n_cols2)

    # --- Panel 3: Difference (intersection only) ---
    vmax_diff = np.nanmax(np.abs(diff_mat)) if np.any(~np.isnan(diff_mat)) else 1.0
    sns.heatmap(
        diff_mat,
        ax=ax3,
        cmap=diff_cmap,
        vmin=-vmax_diff,
        vmax=vmax_diff,
        cbar=False,
        mask=np.isnan(diff_mat),
        xticklabels=False,
        yticklabels=False,
    )
    ax3.set_title("Difference", fontsize=fontsize, fontweight="bold")
    ax3.tick_params(left=False, bottom=False)
    draw_highlight(ax3, gt_row, gt_col_diff, n_rows, n_shared)

    # --- Common axis labels ---
    fig.text(
        0.52,
        0.025,
        "Cell Type",
        ha="center",
        va="bottom",
        fontsize=fontsize,
        fontweight="bold",
    )
    fig.text(
        0.042,
        (bottom + top) / 2,
        "DMR Cell Type Group",
        ha="left",
        va="center",
        rotation=90,
        fontsize=fontsize,
        fontweight="bold",
    )

    # ── 9. Analytical DataFrame ──
    df_dict = {
        "dmr_ctype_label": master_row_indices,
        "dmr_ctype": master_row_names,
        f"{model_names[0]}_target_score": diag1,
        f"{model_names[1]}_target_score": diag2,
        f"difference_({model_names[1]}-{model_names[0]})": diag2 - diag1,
    }
    if true_props_array is not None:
        df_dict["true_proportion"] = true_props_array
        df_dict[f"{model_names[0]}_error"] = diag1 - true_props_array
        df_dict[f"{model_names[1]}_error"] = diag2 - true_props_array
    diag_comparison_df = pd.DataFrame(df_dict)

    if save_path:
        if os.path.dirname(save_path):
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches="tight", dpi=300)
        plt.close(fig)
    else:
        plt.show()

    return diff_mat, diag_comparison_df

In [77]:
pure_ios_hard = hard_pure["test"]
pure_ios_soft = soft_pure["test"]

In [82]:
pure_ios_soft[0].shape

(39, 39)

In [92]:
for i in range(39):
    # Construct a unique filename for each iteration
    file_name = f"comparison_pure_{labels_dict[i]}.png"
    full_path = os.path.join("hard_vs_soft_matrices_latest", file_name)

    print(f"Processing and saving iteration {i}...")
    true_props_array= np.zeros(39)
    true_props_array[i] = 1
    # Call the modified function
    hard_matrix = pd.DataFrame(pure_ios_hard[i])
    soft_matrix = pd.DataFrame(pure_ios_soft[i])
    for df in [hard_matrix, soft_matrix]:
        df.columns = [f"prediction_{x}_wavg" for x in range(df.shape[1])]
        df["dmr_ctype_label"] = range(39)
        df["dmr_ctype"] = df["dmr_ctype_label"].apply(lambda x: labels_dict.get(x))
    compare_model_predictions_v3(
        hard_matrix,
        soft_matrix,
        true_props_array= true_props_array,
        save_path=full_path,
        model_names=[
            "Hard Labeled with Rej. Microbatch balanced (0.5)",
            "Soft Labeled with Jaccard Pooling (d<=0.41)",
        ],
        figsize= (16,4.5)
    )

Processing and saving iteration 0...
Processing and saving iteration 1...
Processing and saving iteration 2...
Processing and saving iteration 3...
Processing and saving iteration 4...
Processing and saving iteration 5...
Processing and saving iteration 6...
Processing and saving iteration 7...
Processing and saving iteration 8...
Processing and saving iteration 9...
Processing and saving iteration 10...
Processing and saving iteration 11...
Processing and saving iteration 12...
Processing and saving iteration 13...
Processing and saving iteration 14...
Processing and saving iteration 15...
Processing and saving iteration 16...
Processing and saving iteration 17...
Processing and saving iteration 18...
Processing and saving iteration 19...
Processing and saving iteration 20...
Processing and saving iteration 21...
Processing and saving iteration 22...
Processing and saving iteration 23...
Processing and saving iteration 24...
Processing and saving iteration 25...
Processing and saving 

### Compare feature selection schemes

In [139]:
from syto.deconvolution.least_squares_deconvolvers import PSLSDeconvolver
from syto.deconvolution.evaluation import compute_deconvolution_metrics

In [ ]:
pseudobulks_hard = reader_hard.read_pseudobulk_matrices("test", 40)
pseudobulks_soft = reader_soft.read_pseudobulk_matrices("test", 39)

In [96]:
pseudobulks_soft = reader_soft.read_pseudobulk_matrices("test", 39)

#### Diagonal Soft

In [ ]:
deconvolver = PSLSDeconvolver()
deconvolver.fit(np.array([np.diag(soft_pure["train"][i]) for i in range(39)]), np.arange(39))
predictions = deconvolver.predict(np.array([np.diag(pseudobulks_soft[0][i]) for i in range(len(pseudobulks_soft[0]))]), n_workers=2)
compute_deconvolution_metrics(predictions, pseudobulks_soft[1])

#### Full Soft

In [149]:
deconvolver = PSLSDeconvolver()
deconvolver.fit(soft_pure["train"].reshape(39, 39*39), np.arange(39))
predictions = deconvolver.predict(pseudobulks_soft[0].reshape(100000, 39*39), n_workers=2)
compute_deconvolution_metrics(predictions, pseudobulks_soft[1])

Predicting with PSLS in parallel: 100%|██████████| 1000/1000 [14:42<00:00,  1.13it/s]


{'mae': 0.0031494238816691797,
 'mse': 0.00016291311279243112,
 'kl': 0.07094190527183979,
 'r2': 0.9803711644784419,
 'max_error': 0.438544350697614,
 'cosine_sim': 0.9909445616752649,
 'loa_lower': -0.025016937783307887,
 'loa_upper': 0.025016937724271965,
 'loa_width': 0.05003387550757985,
 'worst_class_idx': 11,
 'worst_class_name': 11,
 'worst_class_loa_lower': -0.0961921714913126,
 'worst_class_loa_upper': 0.0724651381005054,
 'worst_class_loa_width': 0.168657309591818,
 'per_class_loa': {'bias': array([-2.05376847e-04,  1.57803262e-03,  4.91812580e-03, -1.65952532e-04,
          3.39543172e-03,  4.55662642e-04,  6.82108921e-03,  1.72117125e-03,
          3.58434574e-03, -7.80420081e-04,  2.84153748e-04, -1.18635167e-02,
         -3.01461795e-03, -3.83752176e-03, -1.09492292e-03, -5.35714934e-04,
          5.86289021e-03,  3.55661894e-04, -3.35537660e-04, -6.34529644e-04,
         -5.78012810e-04, -1.45664198e-04, -1.52002801e-03,  1.13574143e-03,
          2.58127191e-03,  2.331

#### Diagonal Hard

In [151]:
deconvolver = PSLSDeconvolver()
deconvolver.fit(np.array([np.diag(hard_pure["train"][i]) for i in range(39)]), np.arange(39))
predictions = deconvolver.predict(np.array([np.diag(pseudobulks_hard[0][i]) for i in range(len(pseudobulks_hard[0]))]), n_workers=2)
compute_deconvolution_metrics(predictions, pseudobulks_hard[1])

Predicting with PSLS in parallel: 100%|██████████| 1000/1000 [01:54<00:00,  8.71it/s]


{'mae': 0.003583650474112441,
 'mse': 0.00016295682754549075,
 'kl': 0.08733721093236017,
 'r2': 0.9803658974395705,
 'max_error': 0.31445528652830795,
 'cosine_sim': 0.9906435750019164,
 'loa_lower': -0.025020294000816002,
 'loa_upper': 0.02502029389402879,
 'loa_width': 0.05004058789484479,
 'worst_class_idx': 11,
 'worst_class_name': 11,
 'worst_class_loa_lower': -0.06745445949864388,
 'worst_class_loa_upper': 0.0736474659273436,
 'worst_class_loa_width': 0.14110192542598748,
 'per_class_loa': {'bias': array([-1.32550381e-04,  7.74333133e-05,  1.97819059e-03, -3.89673535e-04,
          1.54524764e-03,  3.90162111e-04,  5.11349527e-03,  1.10388697e-03,
          2.30015023e-03, -1.61487435e-03,  8.17621321e-05,  3.09650321e-03,
         -2.49684126e-03, -8.45193027e-03, -1.92844794e-03,  1.86591922e-03,
          6.82114794e-03,  1.42955089e-04,  8.09206672e-05,  3.07495466e-03,
         -1.11068364e-03, -1.19683504e-03, -1.64010924e-03,  4.69448900e-04,
          8.17739198e-04, -4.

#### Full Hard

In [150]:
deconvolver = PSLSDeconvolver()
deconvolver.fit(hard_pure["train"].reshape(39, 39*40), np.arange(39))
predictions = deconvolver.predict(pseudobulks_hard[0].reshape(100000, 39*40), n_workers=2)
compute_deconvolution_metrics(predictions, pseudobulks_hard[1])

Predicting with PSLS in parallel: 100%|██████████| 1000/1000 [20:20<00:00,  1.22s/it]


{'mae': 0.0035851016145319703,
 'mse': 0.00016306362487661125,
 'kl': 0.08710012096282241,
 'r2': 0.9803530297998158,
 'max_error': 0.31445624631583,
 'cosine_sim': 0.9906366571645769,
 'loa_lower': -0.025028491461733512,
 'loa_upper': 0.025028491345778052,
 'loa_width': 0.050056982807511564,
 'worst_class_idx': 11,
 'worst_class_name': 11,
 'worst_class_loa_lower': -0.06745762660001915,
 'worst_class_loa_upper': 0.07370659184544613,
 'worst_class_loa_width': 0.14116421844546528,
 'per_class_loa': {'bias': array([-1.32954473e-04,  7.61810800e-05,  1.97325483e-03, -3.89045188e-04,
          1.54247758e-03,  3.88371687e-04,  5.10703219e-03,  1.10614557e-03,
          2.29956042e-03, -1.61502623e-03,  8.15755807e-05,  3.12448262e-03,
         -2.49786954e-03, -8.45767873e-03, -1.92689183e-03,  1.86463103e-03,
          6.82080039e-03,  1.43030796e-04,  7.95986754e-05,  3.07053702e-03,
         -1.11130687e-03, -1.19811327e-03, -1.64184069e-03,  4.69175897e-04,
          8.15821802e-04, -4